06_nonfunctional_explainability.ipynb - Non-Functional & Explainability Testing


In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             accuracy_score, classification_report)
import shap

In [ ]:
# ── 0. Prepare directories ────────────────────────────────────────────
os.makedirs('../results/figures', exist_ok=True)

# ── 1. Load data & model ─────────────────────────────────────────
df    = pd.read_csv('../data/processed/train_feat.csv')
X     = df.drop(columns=['Survived'])
y     = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = joblib.load('../models/randomforest.joblib')
print(f"train: {X_train.shape}  test: {X_test.shape}")


train: (712, 8)  test: (179, 8)


In [ ]:
# ── 1. Load data & model ─────────────────────────────────────────
df    = pd.read_csv('../data/processed/train_feat.csv')
X     = df.drop(columns=['Survived'])
y     = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = joblib.load('../models/randomforest.joblib')
print(f"train: {X_train.shape}  test: {X_test.shape}")


train: (712, 8)  test: (179, 8)


In [ ]:
# ── 2. Resource consumption timing ────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

print("\n── Resource Consumption ──")

# Training time
t0 = time.perf_counter()
tmp_model = RandomForestClassifier(n_estimators=100, random_state=42)
tmp_model.fit(X_train, y_train)
train_time = time.perf_counter() - t0
print(f"[PASS] Training time:  {train_time:.4f} seconds")

# Prediction time (entire test set)
t0 = time.perf_counter()
y_pred = model.predict(X_test)
pred_time = time.perf_counter() - t0
print(f"[PASS] Prediction time:  {pred_time:.6f} seconds  ({len(X_test)} samples)")
print(f"[PASS] Per-sample:  {pred_time/len(X_test)*1000:.4f} ms/sample")

y_prob = model.predict_proba(X_test)[:, 1]
acc    = accuracy_score(y_test, y_pred)
auc    = roc_auc_score(y_test, y_prob)
print(f"\nhold-out  acc={acc:.4f}  auc={auc:.4f}")



── 资源消耗 ──
[PASS] 训练耗时:  0.1151 秒
[PASS] 预测耗时:  0.007318 秒  (179 条样本)
[PASS] 单条预测:  0.0409 ms/sample

hold-out  acc=0.8156  auc=0.8339


In [ ]:
# ── 3. Confusion matrix ────────────────────────────────────────────
print("\n── Confusion Matrix ──")
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix  (acc={acc:.4f})')
plt.tight_layout()
plt.savefig('../results/figures/confusion_matrix.png', dpi=150)
plt.show()
print("[PASS] Saved: results/figures/confusion_matrix.png")



── 混淆矩阵 ──
[PASS] 已保存: results/figures/confusion_matrix.png


In [ ]:
# ── 4. ROC curve ────────────────────────────────────────────
print("\n── ROC Curve ──")
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - RandomForest')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../results/figures/roc_curve.png', dpi=150)
plt.show()
print("[PASS] Saved: results/figures/roc_curve.png")



── ROC 曲线 ──
[PASS] 已保存: results/figures/roc_curve.png


In [ ]:
# ── 5. SHAP explainability ───────────────────────────────────────
print("\n── SHAP Analysis ──")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# 5-1 Summary Plot (feature importance overview)
fig, ax = plt.subplots(figsize=(8, 5))
shap.summary_plot(shap_values[:, :, 1] if shap_values.ndim == 3
                  else shap_values[1],
                  X_test, show=False)
plt.tight_layout()
plt.savefig('../results/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("[PASS] Saved: results/figures/shap_summary.png")

# 5-2 Bar Plot (mean absolute SHAP values)
fig, ax = plt.subplots(figsize=(8, 5))
shap.summary_plot(shap_values[:, :, 1] if shap_values.ndim == 3
                  else shap_values[1],
                  X_test, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig('../results/figures/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("[PASS] Saved: results/figures/shap_bar.png")

# 5-3 Single sample waterfall (first 3)
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = float(expected_value[1])
else:
    expected_value = float(expected_value)

for i in range(3):
    sv = shap_values[i, :, 1] if shap_values.ndim == 3 else shap_values[1][i]
    exp = shap.Explanation(
        values=sv,
        base_values=expected_value,
        data=X_test.iloc[i].values,
        feature_names=X_test.columns.tolist()
    )
    shap.plots.waterfall(exp, show=False)
    plt.tight_layout()
    plt.savefig(f'../results/figures/shap_waterfall_sample{i}.png',
                dpi=150, bbox_inches='tight')
    plt.close()
print("[PASS] Saved: results/figures/shap_waterfall_sample0~2.png")



── SHAP 分析 ──
[PASS] 已保存: results/figures/shap_summary.png
[PASS] 已保存: results/figures/shap_bar.png
[PASS] 已保存: results/figures/shap_waterfall_sample0~2.png


In [ ]:
# ── 6. Assertion tests ────────────────────────────────────────────
print("\n── Assertion tests ──")

assert train_time < 60, \
    f"[FAIL] Training time too long: {train_time:.2f}s"
print(f"[PASS] Training time reasonable: {train_time:.4f}s < 60s")

assert pred_time < 1.0, \
    f"[FAIL] Prediction time too long: {pred_time:.4f}s"
print(f"[PASS] Prediction time reasonable: {pred_time:.6f}s < 1s")

assert auc > 0.75, \
    f"[FAIL] AUC below target: {auc:.4f}"
print(f"[PASS] AUC meets target: {auc:.4f} > 0.75")

for fig_name in ['confusion_matrix.png', 'roc_curve.png',
                 'shap_summary.png', 'shap_bar.png']:
    path = f'../results/figures/{fig_name}'
    assert os.path.exists(path), f"[FAIL] Chart not found: {path}"
print("[PASS] All chart files have been generated")



── 断言测试 ──
[PASS] 训练耗时合理: 0.1151s < 60s
[PASS] 预测耗时合理: 0.007318s < 1s
[PASS] AUC 达标: 0.8339 > 0.75
[PASS] 所有图表文件均已生成


In [ ]:
# ── 7. Archive report ────────────────────────────────────────────
report = classification_report(
    y_test, y_pred, target_names=['Not Survived', 'Survived']
)

with open('../results/nonfunctional_report.txt', 'w', encoding='utf-8') as f:
    f.write("Non-Functional & Explainability Report\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Training time   : {train_time:.4f} s\n")
    f.write(f"Prediction time : {pred_time:.6f} s ({len(X_test)} samples)\n")
    f.write(f"Per-sample      : {pred_time/len(X_test)*1000:.4f} ms\n\n")
    f.write(f"Hold-out Accuracy : {acc:.4f}\n")
    f.write(f"Hold-out AUC      : {auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(report + "\n")
    f.write("Figures saved to: results/figures/\n")
    f.write("  - confusion_matrix.png\n")
    f.write("  - roc_curve.png\n")
    f.write("  - shap_summary.png\n")


[PASS] 报告已归档: results/nonfunctional_report.txt
